In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')
import os
from tqdm import tqdm
import random

# ==========================================
# 1. SET SEED & CONFIGURATION
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = "vinai/phobert-base"
    max_len = 256
    batch_size = 16
    epochs = 5
    learning_rate = 2e-5
    weight_decay = 0.01
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    num_classes = 3  # Sentiment: 0: Negative, 1: Neutral, 2: Positive
    num_topics = 4   # Topic: 0: Lecturer, 1: Training program, 2: Facility, 3: Others
    
    gradient_accumulation_steps = 1
    warmup_ratio = 0.1
    topic_loss_weight = 0.5  # Trọng số loss cho topic (Sentiment quan trọng hơn nên tỷ trọng lớn hơn)
    
config = Config()
print(f"Using device: {config.device}")

# ==========================================
# 2. LOAD DATA
# ==========================================
try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except FileNotFoundError:
    # Fallback paths
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sentiment distribution:\n{train_df['sentiment'].value_counts().sort_index()}")

# ==========================================
# 3. DATASET
# ==========================================
class JointSentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.is_train:
            item['sentiment'] = torch.tensor(self.df.iloc[idx]['sentiment'], dtype=torch.long)
            item['topic'] = torch.tensor(self.df.iloc[idx]['topic'], dtype=torch.long)
            
        return item

# ==========================================
# 4. MODEL (MULTI-TASK CASCADING)
# ==========================================
class JointPhoBERT(nn.Module):
    def __init__(self, model_name, num_topics, num_sentiments):
        super(JointPhoBERT, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        
        # Topic Classifier
        self.topic_classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_topics)
        )
        
        # Sentiment Classifier (Nhận đặc trưng BERT + Xác suất Topic)
        self.sentiment_classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + num_topics, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_sentiments)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        
        # 1. Dự đoán Topic
        topic_logits = self.topic_classifier(pooled_output)
        topic_probs = torch.softmax(topic_logits, dim=1)
        
        # 2. Nối Topic Probs vào Features và dự đoán Sentiment
        combined_features = torch.cat((pooled_output, topic_probs), dim=1)
        sentiment_logits = self.sentiment_classifier(combined_features)
        
        return topic_logits, sentiment_logits

# ==========================================
# 5. TRAINING & EVALUATION FUNCTIONS
# ==========================================
def train_epoch(model, data_loader, optimizer, scheduler, criterion, device):
    model.train()
    losses = []
    all_preds_sent = []
    all_labels_sent = []
    
    progress_bar = tqdm(data_loader, desc='Training', leave=False)
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_sent = batch['sentiment'].to(device)
        labels_topic = batch['topic'].to(device)
        
        optimizer.zero_grad()
        
        topic_logits, sentiment_logits = model(input_ids, attention_mask)
        
        loss_topic = criterion(topic_logits, labels_topic)
        loss_sent = criterion(sentiment_logits, labels_sent)
        
        # Combined Loss
        loss = loss_sent + config.topic_loss_weight * loss_topic
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        losses.append(loss.item())
        
        _, preds_sent = torch.max(sentiment_logits, dim=1)
        all_preds_sent.extend(preds_sent.cpu().numpy())
        all_labels_sent.extend(labels_sent.cpu().numpy())
        
        progress_bar.set_postfix({'loss': np.mean(losses)})
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels_sent, all_preds_sent)
    f1 = f1_score(all_labels_sent, all_preds_sent, average='weighted')
    
    return avg_loss, accuracy, f1

def eval_epoch(model, data_loader, criterion, device):
    model.eval()
    losses = []
    all_preds_sent = []
    all_labels_sent = []
    
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating', leave=False)
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_sent = batch['sentiment'].to(device)
            labels_topic = batch['topic'].to(device)
            
            topic_logits, sentiment_logits = model(input_ids, attention_mask)
            
            loss_topic = criterion(topic_logits, labels_topic)
            loss_sent = criterion(sentiment_logits, labels_sent)
            loss = loss_sent + config.topic_loss_weight * loss_topic
            
            losses.append(loss.item())
            
            _, preds_sent = torch.max(sentiment_logits, dim=1)
            all_preds_sent.extend(preds_sent.cpu().numpy())
            all_labels_sent.extend(labels_sent.cpu().numpy())
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels_sent, all_preds_sent)
    f1 = f1_score(all_labels_sent, all_preds_sent, average='weighted')
    
    return avg_loss, accuracy, f1

def predict(model, data_loader, device):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Quá trình Forward pass lấy trực tiếp sentiment
            _, sentiment_logits = model(input_ids, attention_mask)
            _, preds = torch.max(sentiment_logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    return all_preds

# ==========================================
# 6. MAIN EXECUTION & CROSS VALIDATION
# ==========================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

# Prepare Test Data
test_dataset = JointSentimentDataset(test_df, tokenizer, config.max_len, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_predictions = []
val_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*50}")
    print(f"Fold {fold + 1}/{config.n_folds}")
    print(f"{'='*50}")
    
    # Split data
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Create datasets
    train_dataset = JointSentimentDataset(train_fold, tokenizer, config.max_len, is_train=True)
    val_dataset = JointSentimentDataset(val_fold, tokenizer, config.max_len, is_train=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    # Initialize Joint Model
    print(f"Initializing Multi-Task model for fold {fold + 1}...")
    model = JointPhoBERT(config.model_name, config.num_topics, config.num_classes).to(config.device)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Training Loop
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        
        train_loss, train_acc, train_f1 = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, config.device
        )
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, config.device)
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"--> New best model! Val F1: {val_f1:.4f}")
    
    # Load best model & Predict on test set
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    print(f"Predicting on test set for fold {fold + 1}...")
    fold_pred = predict(model, test_loader, config.device)
    fold_predictions.append(fold_pred)

print(f"\nCross-validation scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")

# ==========================================
# 7. ENSEMBLE & SUBMISSION
# ==========================================
final_predictions = []
fold_predictions = np.array(fold_predictions)

# Majority Voting
for i in range(fold_predictions.shape[1]):
    pred_counts = np.bincount(fold_predictions[:, i])
    final_predictions.append(np.argmax(pred_counts))

submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("\nSubmission saved to /kaggle/working/submission.csv")
print("\nSentiment distribution in predictions:")
print(submission['sentiment'].value_counts().sort_index())

# Weighted Ensemble
if len(val_scores) > 0:
    weights = np.array(val_scores) / np.sum(val_scores)
    
    weighted_predictions = []
    for i in range(fold_predictions.shape[1]):
        weighted_votes = np.zeros(config.num_classes)
        for fold_idx in range(len(fold_predictions)):
            weighted_votes[fold_predictions[fold_idx, i]] += weights[fold_idx]
        weighted_predictions.append(np.argmax(weighted_votes))
    
    submission_weighted = pd.DataFrame({
        'id': test_df['id'],
        'sentiment': weighted_predictions
    })
    submission_weighted.to_csv('/kaggle/working/submission_weighted.csv', index=False)
    print("Weighted ensemble submission saved to /kaggle/working/submission_weighted.csv")

print("\nDone!")

Using device: cuda
Train shape: (11322, 4)
Test shape: (4853, 2)
Sentiment distribution:
sentiment
0    5226
1     501
2    5595
Name: count, dtype: int64
Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Val   - Loss: 0.4463, Acc: 0.9333, F1: 0.9307

Epoch 5/5


Train - Loss: 0.2290, Acc: 0.9728, F1: 0.9723


Val   - Loss: 0.4438, Acc: 0.9315, F1: 0.9298
Predicting on test set for fold 5...


Predicting: 100%|██████████| 304/304 [01:00<00:00,  5.04it/s]


Cross-validation scores: [0.9426448161630696, 0.9386764616756481, 0.9334551099105138, 0.9366705994679678, 0.9310043138294072]
Mean CV F1: 0.9365 (+/- 0.0040)

Submission saved to /kaggle/working/submission.csv

Sentiment distribution in predictions:
sentiment
0    2252
1     132
2    2469
Name: count, dtype: int64
Weighted ensemble submission saved to /kaggle/working/submission_weighted.csv

Done!
